# **Stock Market Sector wise Data Collection**
- Indain stock market
- 2018-2026(Because it cover all the major geopolitical events)
- More about yfinance symbols -> https://finance.yahoo.com/lookup/

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

# 1. **Define NSE Sector Indices**

In [2]:
sector_indices = {
    "NIFTY_BANK": "^NSEBANK",
    "NIFTY_IT": "^CNXIT",
    "NIFTY_AUTO": "^CNXAUTO",
    "NIFTY_PHARMA": "^CNXPHARMA",
    "NIFTY_FMCG": "^CNXFMCG",
    "NIFTY_METAL": "^CNXMETAL",
    "NIFTY_ENERGY": "^CNXENERGY",
    "NIFTY_REALTY": "^CNXREALTY",
    "NIFTY50":"^NSEI"
}

## 2. **Download Historical Data**

In [12]:
start_date = "2018-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

#dictionary to store the dataframes 
sector_data = {}

for sector,ticker in sector_indices.items():
    df = yf.download(ticker,
                    start=start_date,
                    end=end_date, 
                    auto_adjust=True, 
                    progress=False)

    #Remove missing rows
    df = df.dropna()
    #Store the dataframe in the dictionary
    sector_data[sector] = df

    #Print the shape of the dataframe for each sector
    print(f"{sector}: {df.shape}")

    #Print the first few rows of the dataframe for each sector
    print(f"{sector}:\n{df.head()}\n")

    #Save individual csv
    csv_path = f"E:/BDA/Summer 2026/Data/{sector}_data.csv" 
    df.to_csv(csv_path)

    print(f"Saved {sector} data to {csv_path}\n")
    print(f"Shape:{df.shape}")
    print("-" * 50)
print("All data downloaded and saved successfully!")

NIFTY_BANK: (2086, 5)
NIFTY_BANK:
Price              Close          High           Low          Open   Volume
Ticker          ^NSEBANK      ^NSEBANK      ^NSEBANK      ^NSEBANK ^NSEBANK
Date                                                                       
2018-01-01  25317.804688  25587.701934  25271.256402  25565.452193    57600
2018-01-02  25337.955078  25425.204063  25232.507087  25381.903785        0
2018-01-03  25318.304688  25454.603881  25300.605675  25425.453830        0
2018-01-04  25462.302734  25490.052411  25310.005682  25367.354623        0
2018-01-05  25601.552734  25643.052253  25499.255092  25524.153241        0

Saved NIFTY_BANK data to E:/BDA/Summer 2026/Data/NIFTY_BANK_data.csv

Shape:(2086, 5)
--------------------------------------------------
NIFTY_IT: (2086, 5)
NIFTY_IT:
Price              Close          High           Low          Open Volume
Ticker            ^CNXIT        ^CNXIT        ^CNXIT        ^CNXIT ^CNXIT
Date                                      

# 3. **Combain Datasets**

In [13]:
close_prices = pd.DataFrame()

for sector,df in sector_data.items():
    close_prices[sector] = df['Close']

    # Handling missing values
    close_prices = close_prices.ffill().bfill()
# Save the combined close prices to a CSV file
combined_csv_path = "E:/BDA/Summer 2026/Data/combined_sector_close_prices.csv"
close_prices.to_csv(combined_csv_path)
print(f"Combined close prices saved to {combined_csv_path}")
print(f"Combined close prices sample:\n{close_prices.head()}")
print(f"Shape of combined close prices: {close_prices.shape}")

Combined close prices saved to E:/BDA/Summer 2026/Data/combined_sector_close_prices.csv
Combined close prices sample:
              NIFTY_BANK      NIFTY_IT    NIFTY_AUTO  NIFTY_PHARMA  \
Date                                                                 
2018-01-01  25317.804688  11565.750000  11890.500000   9618.150391   
2018-01-02  25337.955078  11602.349609  11927.150391   9565.799805   
2018-01-03  25318.304688  11586.700195  11866.799805   9519.549805   
2018-01-04  25462.302734  11590.799805  11843.250000   9633.599609   
2018-01-05  25601.552734  11633.549805  11929.599609   9701.049805   

              NIFTY_FMCG  NIFTY_METAL  NIFTY_ENERGY  NIFTY_REALTY  \
Date                                                                
2018-01-01  26686.349609  3919.199951  14138.950195    347.700012   
2018-01-02  26533.500000  3942.850098  14162.400391    343.850006   
2018-01-03  26648.400391  4000.199951  14157.799805    345.450012   
2018-01-04  26689.599609  4112.000000  14247.2

## 3. **Feature Engineering**

In [4]:
combined_csv_path = r"E:\BDA\Summer 2026\Data\raw_data\sector_raw_data\processed_engenieard_data\combined_sector_close_prices.csv"
df_combined = pd.read_csv(combined_csv_path, index_col=0, parse_dates=True)
df_combined.head()

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50
Date,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609


In [5]:
df_featured_data = df_combined.copy()
sector_columns = list(sector_indices.keys())
df_featured_data.sample(5)

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50
Date,,,,,,,,,
2019-02-19,26684.539062,15333.500000,8073.399902,8552.700195,29009.699219,2689.699951,14411.049805,228.100006,10604.349609
2021-10-27,40873.875000,35503.898438,11395.150391,14094.250000,38946.101562,5741.899902,24184.849609,515.950012,18210.949219
2020-08-18,22170.341797,18270.349609,7941.850098,11581.700195,31726.150391,2507.500000,15634.349609,217.600006,11385.349609
2024-07-22,52280.398438,39773.148438,25250.750000,20641.550781,60881.750000,9350.599609,41945.500000,1090.900024,24509.250000
2022-02-17,37531.210938,34634.101562,11492.849609,13324.400391,36571.699219,5793.399902,24757.949219,454.700012,17304.599609


### **Calculate daily return**

In [6]:
#Return
for sector in sector_columns:
    df_featured_data[f"{sector}_Return"] = df_featured_data[sector].pct_change()

df_featured_data.head()

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,NIFTY_IT_Return,NIFTY_AUTO_Return,NIFTY_PHARMA_Return,NIFTY_FMCG_Return,NIFTY_METAL_Return,NIFTY_ENERGY_Return,NIFTY_REALTY_Return,NIFTY50_Return
Date,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,0.003164,0.003082,-0.005443,-0.005728,0.006034,0.001659,-0.011073,0.000000
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,-0.001349,-0.005060,-0.004835,0.004330,0.014545,-0.000325,0.004653,0.000096
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,0.000354,-0.001985,0.011981,0.001546,0.027949,0.006318,-0.001882,0.005899
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,0.003688,0.007291,0.007002,0.008419,0.004475,0.000186,0.007251,0.005145


### **Calculate log return**

In [8]:
#log return
for sector in sector_columns:
    df_featured_data[f"{sector}_Log_Return"] = np.log(
        df_featured_data[sector] /
        df_featured_data[sector].shift(1)
    )

df_featured_data

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY50_Return,NIFTY_BANK_Log_Return,NIFTY_IT_Log_Return,NIFTY_AUTO_Log_Return,NIFTY_PHARMA_Log_Return,NIFTY_FMCG_Log_Return,NIFTY_METAL_Log_Return,NIFTY_ENERGY_Log_Return,NIFTY_REALTY_Log_Return,NIFTY50_Log_Return
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,...,0.000000,0.000796,0.003159,0.003078,-0.005458,-0.005744,0.006016,0.001657,-0.011135,0.000000
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,...,0.000096,-0.000776,-0.001350,-0.005073,-0.004847,0.004321,0.014441,-0.000325,0.004642,0.000096
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,...,0.005899,0.005671,0.000354,-0.001986,0.011909,0.001545,0.027565,0.006298,-0.001883,0.005881
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,...,0.005145,0.005454,0.003681,0.007265,0.006977,0.008384,0.004465,0.000186,0.007224,0.005132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-15,57198.800781,28067.900391,26977.500000,24220.099609,49043.398438,13083.849609,39493.949219,800.049988,23853.900391,0.006759,...,0.009779,0.006736,0.009743,0.025668,-0.006582,0.004410,0.017685,0.006337,0.038803,0.009731
2026-06-16,57297.148438,28568.099609,26891.550781,24157.849609,49642.398438,12881.500000,39915.398438,818.099976,23989.150391,0.001719,...,0.005670,0.001718,0.017664,-0.003191,-0.002573,0.012140,-0.015586,0.010615,0.022310,0.005654
2026-06-17,57585.050781,28809.550781,26724.400391,24146.550781,49556.050781,13011.400391,40238.449219,814.599976,24085.699219,0.005025,...,0.004025,0.005012,0.008416,-0.006235,-0.000468,-0.001741,0.010034,0.008061,-0.004287,0.004017


### **Calculate Volatility**

what is volitility

why 30 day rolling window 
- If you use too few days (like 5), volatility looks very jumpy.
- If you use too many days (like 180), volatility becomes too smooth and slow to react.
- 30 days is a good middle ground: responsive but not overly noisy.

In [9]:
# Volatility
for sector in sector_columns:
    df_featured_data[f"{sector}_7_Volatility"] = df_featured_data[f"{sector}_Return"].rolling(window=7).std()
    df_featured_data[f"{sector}_30_Volatility"] = df_featured_data[f"{sector}_Return"].rolling(window=30).std()
df_featured_data.head()

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY_FMCG_7_Volatility,NIFTY_FMCG_30_Volatility,NIFTY_METAL_7_Volatility,NIFTY_METAL_30_Volatility,NIFTY_ENERGY_7_Volatility,NIFTY_ENERGY_30_Volatility,NIFTY_REALTY_7_Volatility,NIFTY_REALTY_30_Volatility,NIFTY50_7_Volatility,NIFTY50_30_Volatility
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### **Calculate Moving Averages**

In [10]:
# Moving Average
for sector in sector_columns:
    df_featured_data[f"{sector}_MA_7"] = df_featured_data[sector].rolling(window=7).mean()
    df_featured_data[f"{sector}_MA_30"] = df_featured_data[sector].rolling(window=30).mean()
    df_featured_data[f"{sector}_MA_50"] = df_featured_data[sector].rolling(window=50).mean()

df_featured_data.head()

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY_METAL_MA_50,NIFTY_ENERGY_MA_7,NIFTY_ENERGY_MA_30,NIFTY_ENERGY_MA_50,NIFTY_REALTY_MA_7,NIFTY_REALTY_MA_30,NIFTY_REALTY_MA_50,NIFTY50_MA_7,NIFTY50_MA_30,NIFTY50_MA_50
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### **Calculate Momentum**

Momentum is the rate of change in prices or returns — it captures whether a sector or stock is trending upward or downward and how strong that trend is.
- Momentum = Current pricr - price in n days ago
- Return Momentum = (Current pricr /price in n days ago) - 1

In [11]:
# Momentum
for sector in sector_columns:
    df_featured_data[f"{sector}_Momentum"] = (df_featured_data[sector] / df_featured_data[sector].shift(10) ) - 1

df_featured_data.head()

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY50_MA_50,NIFTY_BANK_Momentum,NIFTY_IT_Momentum,NIFTY_AUTO_Momentum,NIFTY_PHARMA_Momentum,NIFTY_FMCG_Momentum,NIFTY_METAL_Momentum,NIFTY_ENERGY_Momentum,NIFTY_REALTY_Momentum,NIFTY50_Momentum
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### **Calculate Relative Strength**

Relative Strength (RS) is a way to measure how one asset (like a sector index) is performing compared to another benchmark (like NIFTY50). It’s often used to identify which sectors are outperforming or underperforming the broader market.

- 𝑅𝑆 =Sector Index/Benchmark Index
- - If RS > 1 → the sector is stronger than the benchmark.

- - If RS < 1 → the sector is weaker than the benchmark.


" Relative Strength compares sector performance to a benchmark like NIFTY50, helping you spot outperformers and laggards."

In [12]:
# Relative Strength 
for sector in sector_columns:
    df_featured_data[f"{sector}_RS"] = df_featured_data[sector]/df_featured_data['NIFTY50']

df_featured_data.head()

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY50_Momentum,NIFTY_BANK_RS,NIFTY_IT_RS,NIFTY_AUTO_RS,NIFTY_PHARMA_RS,NIFTY_FMCG_RS,NIFTY_METAL_RS,NIFTY_ENERGY_RS,NIFTY_REALTY_RS,NIFTY50_RS
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,...,NaN,2.424566,1.107597,1.138697,0.921085,2.555625,0.375323,1.354020,0.033298,1.0
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,...,NaN,2.426496,1.111102,1.142207,0.916071,2.540987,0.377588,1.356266,0.032929,1.0
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,...,NaN,2.424382,1.109497,1.136318,0.911555,2.551747,0.383043,1.355696,0.033079,1.0
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,...,NaN,2.423873,1.103381,1.127413,0.917066,2.540705,0.391440,1.356261,0.032823,1.0
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,...,NaN,2.424654,1.101782,1.129820,0.918760,2.548980,0.391179,1.349569,0.032892,1.0


### **Calculate RSI**

The Relative Strength Index (RSI) is a popular momentum oscillator used in technical analysis. Unlike the simple Relative Strength (sector vs. benchmark ratio), RSI measures the speed and magnitude of recent price changes to identify whether a stock or sector is overbought or oversold.
- Relative Strength (RS):

$$
𝑅𝑆=Average Gain over n periods/Average Loss over n periods

𝑅𝑆𝐼=100−100/1+𝑅𝑆

$$
- - RSI > 70 → Overbought (price may be due for a pullback).

- - RSI < 30 → Oversold (price may be due for a rebound).

- - RSI ~ 50 → Neutral trend.



In [13]:
# Relative Strength Index (RSI)
for sector in sector_columns:
    delta = df_featured_data[sector].diff()
    gain = delta.where(delta > 0,0)
    loss = -delta.where(delta < 0,0)
    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    rs = avg_gain / avg_loss
    df_featured_data[f"{sector}_RSI"] = 100 - (100 / (1 + rs))

df_featured_data.head()

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY50_RS,NIFTY_BANK_RSI,NIFTY_IT_RSI,NIFTY_AUTO_RSI,NIFTY_PHARMA_RSI,NIFTY_FMCG_RSI,NIFTY_METAL_RSI,NIFTY_ENERGY_RSI,NIFTY_REALTY_RSI,NIFTY50_RSI
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Calculate Drawdown
- drawdown tells you how much value has fallen from the highest point so far, expressed as a percentage.
$
Drawdown = (Current Value − Peak Value)/Peak Value
$


In [21]:
for sector in sector_columns:

    rolling_max = df_featured_data[sector].cummax()

    df_featured_data[f"{sector}_Drawdown"] = (
        df_featured_data[sector] - rolling_max
    ) / rolling_max

print(df_featured_data.index)

print(df_featured_data.index.name)


df_featured_data

RangeIndex(start=0, stop=2037, step=1)
None


,index,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,...,NIFTY50_Drawdown,NIFTY_BANK_Price_MA30,NIFTY_IT_Price_MA30,NIFTY_AUTO_Price_MA30,NIFTY_PHARMA_Price_MA30,NIFTY_FMCG_Price_MA30,NIFTY_METAL_Price_MA30,NIFTY_ENERGY_Price_MA30,NIFTY_REALTY_Price_MA30,NIFTY50_Price_MA30
0,0,24851.361328,12786.950195,11060.599609,8701.549805,26532.150391,3765.350098,13843.250000,320.299988,10410.900391,...,0.000000,0.978984,1.007036,0.991829,0.968125,0.996449,0.958313,1.001267,0.986626,0.991025
1,1,24791.560547,12762.299805,11051.750000,8670.549805,26381.300781,3753.600098,13667.150391,319.950012,10360.150391,...,-0.004875,0.979815,1.006007,0.992745,0.967881,0.992188,0.958271,0.990204,0.988069,0.988356
2,2,24489.265625,12698.400391,10861.400391,8546.799805,26052.150391,3664.500000,13419.250000,314.799988,10195.150391,...,-0.020723,0.971567,1.001728,0.977844,0.957051,0.981131,0.938730,0.974679,0.975166,0.975197
3,3,24244.818359,12421.299805,10787.000000,8512.900391,25935.800781,3566.800049,13324.000000,305.549988,10094.250000,...,-0.030415,0.965669,0.981271,0.973795,0.955559,0.978489,0.917657,0.969968,0.950078,0.968394
4,4,24167.917969,12609.099609,10848.599609,8578.700195,26044.000000,3555.449951,13244.650391,304.299988,10124.349609,...,-0.027524,0.965532,0.997090,0.980694,0.964619,0.983977,0.917829,0.965435,0.947813,0.973262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,2032,57198.800781,28067.900391,26977.500000,24220.099609,49043.398438,13083.849609,39493.949219,800.049988,23853.900391,...,-0.093991,1.047959,0.973442,1.028482,0.996621,0.982981,0.991530,0.983426,1.029314,1.008051
2033,2033,57297.148438,28568.099609,26891.550781,24157.849609,49642.398438,12881.500000,39915.398438,818.099976,23989.150391,...,-0.088854,1.048212,0.991372,1.024081,0.993130,0.996067,0.976420,0.994791,1.052293,1.013952
2034,2034,57585.050781,28809.550781,26724.400391,24146.550781,49556.050781,13011.400391,40238.449219,814.599976,24085.699219,...,-0.085187,1.051531,1.000096,1.017029,0.991880,0.995691,0.986212,1.003469,1.047192,1.017957
2035,2035,57963.800781,28466.449219,26746.400391,24282.199219,49654.500000,13010.400391,40457.300781,820.200012,24168.000000,...,-0.082061,1.057172,0.989103,1.017965,0.997218,0.998878,0.986433,1.009220,1.054486,1.021670


### Calculating Price/MA Ratio
- feature tells you how far each sector’s price is relative to its 30-day moving average, acting as a trend strength signal.

- - Ratio > 1 (Bullish)
- - Ratio < 1 (Bearish)
- - Ratio ≈ 1 (Neutral)

In [15]:
for sector in sector_columns:

    df_featured_data[f"{sector}_Price_MA30"] = (
        df_featured_data[sector] /
        df_featured_data[f"{sector}_MA_30"]
    )

df_featured_data

C:\Users\dell\AppData\Local\Temp\ipykernel_14812\3856383658.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_featured_data[f"{sector}_Price_MA30"] = (
C:\Users\dell\AppData\Local\Temp\ipykernel_14812\3856383658.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_featured_data[f"{sector}_Price_MA30"] = (
C:\Users\dell\AppData\Local\Temp\ipykernel_14812\3856383658.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY50_Drawdown,NIFTY_BANK_Price_MA30,NIFTY_IT_Price_MA30,NIFTY_AUTO_Price_MA30,NIFTY_PHARMA_Price_MA30,NIFTY_FMCG_Price_MA30,NIFTY_METAL_Price_MA30,NIFTY_ENERGY_Price_MA30,NIFTY_REALTY_Price_MA30,NIFTY50_Price_MA30
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,25317.804688,11565.750000,11890.500000,9618.150391,26686.349609,3919.199951,14138.950195,347.700012,10442.200195,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02,25337.955078,11602.349609,11927.150391,9565.799805,26533.500000,3942.850098,14162.400391,343.850006,10442.200195,0.000796,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,25318.304688,11586.700195,11866.799805,9519.549805,26648.400391,4000.199951,14157.799805,345.450012,10443.200195,-0.000776,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,25462.302734,11590.799805,11843.250000,9633.599609,26689.599609,4112.000000,14247.250000,344.799988,10504.799805,0.005688,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,25601.552734,11633.549805,11929.599609,9701.049805,26914.300781,4130.399902,14249.900391,347.299988,10558.849609,0.005469,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-15,57198.800781,28067.900391,26977.500000,24220.099609,49043.398438,13083.849609,39493.949219,800.049988,23853.900391,0.006759,...,-0.093991,1.047959,0.973442,1.028482,0.996621,0.982981,0.991530,0.983426,1.029314,1.008051
2026-06-16,57297.148438,28568.099609,26891.550781,24157.849609,49642.398438,12881.500000,39915.398438,818.099976,23989.150391,0.001719,...,-0.088854,1.048212,0.991372,1.024081,0.993130,0.996067,0.976420,0.994791,1.052293,1.013952
2026-06-17,57585.050781,28809.550781,26724.400391,24146.550781,49556.050781,13011.400391,40238.449219,814.599976,24085.699219,0.005025,...,-0.085187,1.051531,1.000096,1.017029,0.991880,0.995691,0.986212,1.003469,1.047192,1.017957


In [18]:
print(df_featured_data.index)

df_featured_data = df_featured_data.dropna()
print(df_featured_data.index)

df_featured_data = df_featured_data.reset_index()
print(df_featured_data.columns)

RangeIndex(start=0, stop=2037, step=1)
RangeIndex(start=0, stop=2037, step=1)
Index(['index', 'NIFTY_BANK', 'NIFTY_IT', 'NIFTY_AUTO', 'NIFTY_PHARMA',
       'NIFTY_FMCG', 'NIFTY_METAL', 'NIFTY_ENERGY', 'NIFTY_REALTY', 'NIFTY50',
       ...
       'NIFTY50_Drawdown', 'NIFTY_BANK_Price_MA30', 'NIFTY_IT_Price_MA30',
       'NIFTY_AUTO_Price_MA30', 'NIFTY_PHARMA_Price_MA30',
       'NIFTY_FMCG_Price_MA30', 'NIFTY_METAL_Price_MA30',
       'NIFTY_ENERGY_Price_MA30', 'NIFTY_REALTY_Price_MA30',
       'NIFTY50_Price_MA30'],
      dtype='object', length=118)


### **Droping nan values**

In [17]:
df_featured_data = (
    df_featured_data
    .dropna()
    .reset_index(drop=True)
)
df_featured_data

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY50_Drawdown,NIFTY_BANK_Price_MA30,NIFTY_IT_Price_MA30,NIFTY_AUTO_Price_MA30,NIFTY_PHARMA_Price_MA30,NIFTY_FMCG_Price_MA30,NIFTY_METAL_Price_MA30,NIFTY_ENERGY_Price_MA30,NIFTY_REALTY_Price_MA30,NIFTY50_Price_MA30
0,24851.361328,12786.950195,11060.599609,8701.549805,26532.150391,3765.350098,13843.250000,320.299988,10410.900391,0.004568,...,-0.064643,0.978984,1.007036,0.991829,0.968125,0.996449,0.958313,1.001267,0.986626,0.991025
1,24791.560547,12762.299805,11051.750000,8670.549805,26381.300781,3753.600098,13667.150391,319.950012,10360.150391,-0.002406,...,-0.069202,0.979815,1.006007,0.992745,0.967881,0.992188,0.958271,0.990204,0.988069,0.988356
2,24489.265625,12698.400391,10861.400391,8546.799805,26052.150391,3664.500000,13419.250000,314.799988,10195.150391,-0.012193,...,-0.084027,0.971567,1.001728,0.977844,0.957051,0.981131,0.938730,0.974679,0.975166,0.975197
3,24244.818359,12421.299805,10787.000000,8512.900391,25935.800781,3566.800049,13324.000000,305.549988,10094.250000,-0.009982,...,-0.093092,0.965669,0.981271,0.973795,0.955559,0.978489,0.917657,0.969968,0.950078,0.968394
4,24167.917969,12609.099609,10848.599609,8578.700195,26044.000000,3555.449951,13244.650391,304.299988,10124.349609,-0.003172,...,-0.090388,0.965532,0.997090,0.980694,0.964619,0.983977,0.917829,0.965435,0.947813,0.973262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,57198.800781,28067.900391,26977.500000,24220.099609,49043.398438,13083.849609,39493.949219,800.049988,23853.900391,0.006759,...,-0.093991,1.047959,0.973442,1.028482,0.996621,0.982981,0.991530,0.983426,1.029314,1.008051
2033,57297.148438,28568.099609,26891.550781,24157.849609,49642.398438,12881.500000,39915.398438,818.099976,23989.150391,0.001719,...,-0.088854,1.048212,0.991372,1.024081,0.993130,0.996067,0.976420,0.994791,1.052293,1.013952
2034,57585.050781,28809.550781,26724.400391,24146.550781,49556.050781,13011.400391,40238.449219,814.599976,24085.699219,0.005025,...,-0.085187,1.051531,1.000096,1.017029,0.991880,0.995691,0.986212,1.003469,1.047192,1.017957
2035,57963.800781,28466.449219,26746.400391,24282.199219,49654.500000,13010.400391,40457.300781,820.200012,24168.000000,0.006577,...,-0.082061,1.057172,0.989103,1.017965,0.997218,0.998878,0.986433,1.009220,1.054486,1.021670


# **Save new dataset with new featured**

In [31]:
df_featured_data.to_csv("E:/BDA/Summer 2026/Data/featured_sector_data.csv", index=False)
df_featured_data.sample(5)

,NIFTY_BANK,NIFTY_IT,NIFTY_AUTO,NIFTY_PHARMA,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_REALTY,NIFTY50,NIFTY_BANK_Return,...,NIFTY50_RS,NIFTY_BANK_RSI,NIFTY_IT_RSI,NIFTY_AUTO_RSI,NIFTY_PHARMA_RSI,NIFTY_FMCG_RSI,NIFTY_METAL_RSI,NIFTY_ENERGY_RSI,NIFTY_REALTY_RSI,NIFTY50_RSI
Date,,,,,,,,,,,,,,,,,,,,,
2023-03-27,39431.300781,27948.750000,12035.099609,11882.750000,45188.000000,5372.850098,22514.550781,376.549988,16985.699219,0.000913,...,1.0,-75.366434,-18.065837,-23.661974,561.593560,-436.100204,-85.637481,-390.978106,-48.048022,-51.670685
2020-11-13,28465.369141,21436.849609,8308.900391,11726.450195,31222.099609,2681.000000,15869.250000,241.800003,12719.950195,0.006609,...,1.0,132.681461,965.670041,141.746386,339.898024,154.077890,122.306508,143.287123,262.331623,132.067898
2019-12-19,32241.074219,15812.950195,8165.500000,8073.049805,30386.099609,2691.100098,16112.900391,290.399994,12259.700195,-0.000087,...,1.0,504.453864,188.924907,669.245037,-235.927606,-140.456237,426.828261,-593.626343,217.964059,245.838169
2024-10-17,51288.800781,42734.351562,24991.199219,23254.000000,61352.750000,9603.299805,40983.000000,1044.150024,24749.849609,-0.009889,...,1.0,-58.307779,486.204054,-36.687910,47244.675488,-17.684667,-98.195506,-39.580103,-87.492804,-27.117671
2025-10-07,56239.351562,34709.449219,26933.550781,21875.300781,54763.851562,10156.650391,35470.000000,891.700012,25108.300781,0.002397,...,1.0,243.173589,-106.844454,-405.520330,-222.556399,-52.416441,336.898778,4821.870776,-175.687582,-521.182935


In [32]:
df_featured_data.columns.tolist()

['NIFTY_BANK',
 'NIFTY_IT',
 'NIFTY_AUTO',
 'NIFTY_PHARMA',
 'NIFTY_FMCG',
 'NIFTY_METAL',
 'NIFTY_ENERGY',
 'NIFTY_REALTY',
 'NIFTY50',
 'NIFTY_BANK_Return',
 'NIFTY_IT_Return',
 'NIFTY_AUTO_Return',
 'NIFTY_PHARMA_Return',
 'NIFTY_FMCG_Return',
 'NIFTY_METAL_Return',
 'NIFTY_ENERGY_Return',
 'NIFTY_REALTY_Return',
 'NIFTY50_Return',
 'NIFTY_BANK_Volatility',
 'NIFTY_IT_Volatility',
 'NIFTY_AUTO_Volatility',
 'NIFTY_PHARMA_Volatility',
 'NIFTY_FMCG_Volatility',
 'NIFTY_METAL_Volatility',
 'NIFTY_ENERGY_Volatility',
 'NIFTY_REALTY_Volatility',
 'NIFTY50_Volatility',
 'NIFTY_BANK_MA_7',
 'NIFTY_BANK_MA_30',
 'NIFTY_IT_MA_7',
 'NIFTY_IT_MA_30',
 'NIFTY_AUTO_MA_7',
 'NIFTY_AUTO_MA_30',
 'NIFTY_PHARMA_MA_7',
 'NIFTY_PHARMA_MA_30',
 'NIFTY_FMCG_MA_7',
 'NIFTY_FMCG_MA_30',
 'NIFTY_METAL_MA_7',
 'NIFTY_METAL_MA_30',
 'NIFTY_ENERGY_MA_7',
 'NIFTY_ENERGY_MA_30',
 'NIFTY_REALTY_MA_7',
 'NIFTY_REALTY_MA_30',
 'NIFTY50_MA_7',
 'NIFTY50_MA_30',
 'NIFTY_BANK_Momentum',
 'NIFTY_IT_Momentum',
 'NIFTY_